In [ ]:
# ===============================
# 1. BASIC SETUP & LIBRARIES
# ===============================

import pandas as pd
import numpy as np
import re
import warnings

import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

warnings.filterwarnings("ignore")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
file_path = "/content/drive/MyDrive/Database Customer by Sales 2.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"File tidak ditemukan di path: {file_path}")

df = pd.read_csv(file_path, encoding="latin1")

print("Data shape:", df.shape)
df.head()

In [ ]:
df.info()
df.head(3)
df.tail(3)


In [ ]:
df.head()

In [ ]:
# Asumsi df sudah berhasil dibaca dengan encoding latin1
df = df.copy()


In [ ]:
# Rename kolom agar konsisten dan mudah dibaca
df.columns = [
    "no",
    "company_name",
    "segment",
    "address",
    "contact_person",
    "job_title",
    "phone",
    "email"
]


##Cleaning Dasar (whitespace & karakter aneh)

In [ ]:
# Menghapus spasi aneh dan karakter non-breaking space
df = df.applymap(
    lambda x: x.replace('\xa0', ' ').strip() if isinstance(x, str) else x
)


##Normalisasi Segment (Critical untuk Mapping)

In [ ]:
# Standarisasi segment
df["segment"] = df["segment"].str.lower().str.strip()

segment_mapping = {
    "lab service": "Lab Service",
    "farmasi": "Pharmaceutical",
    "oil": "Oil & Energy",
    "petrochemical": "Petrochemical",
    "paint & coating": "Paint & Coating",
    "health": "Health",
    "agro industry": "Agro Industry",
    "food & beverages": "Food & Beverages"
}

df["segment_clean"] = df["segment"].map(segment_mapping).fillna("Others")


#Cek Data Duplikat

In [ ]:
#Duplicate by company_name
dup_company = (
    df[df.duplicated(subset=["company_name"], keep=False)]
    .sort_values("company_name")
)

print("Jumlah baris duplikat berdasarkan company_name:", dup_company.shape[0])
dup_company["company_name"].value_counts().head(100)


In [ ]:
#Duplicate by (company_name + job_title)
dup_company_job = (
    df[df.duplicated(subset=["company_name", "job_title"], keep=False)]
    .sort_values(["company_name", "job_title"])
)

print("Jumlah baris duplikat berdasarkan company_name + job_title:", dup_company_job.shape[0])
dup_company_job[["company_name", "job_title"]].value_counts().head(10)


In [ ]:
dup_company_job[["company_name", "job_title"]].value_counts()


In [ ]:
dup_company_job[["company_name", "job_title"]].value_counts().sum()


In [ ]:
dup_company_job


In [ ]:
#Duplicate by (company_name + address)
dup_company_address = (
    df[df.duplicated(subset=["company_name", "address"], keep=False)]
    .sort_values(["company_name", "address"])
)

print("Jumlah baris duplikat berdasarkan company_name + address:", dup_company_address.shape[0])
dup_company_address[["company_name", "address"]].value_counts().head(70)


In [ ]:
#company_name + address + job_title  → identik
# Definisi Repeated Valid Entry
# Jika:
# company_name sama
# tapi address ATAU job_title berbeda
# ➡️ bukan duplikat, melainkan entitas/role berbeda

#Identifikasi duplikat berdasarkan 3 kolom

dup_strict = (
    df[df.duplicated(
        subset=["company_name", "address", "job_title"],
        keep=False
    )]
    .sort_values(["company_name", "address", "job_title"])
)

print("Jumlah baris duplikat murni (3 kolom):", dup_strict.shape[0])



In [ ]:
dup_summary = (
    dup_strict
    .groupby(["company_name", "address", "job_title"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

dup_summary


#Handling Duplikat ditunda sampai selesai preprocessing

In [ ]:
# REFINED SEGMENT CLEANING (DATA-DRIVEN, MINIM OTHERS)
# =========================================================
# Dasar perbaikan:
# - Mapping dibuat berdasarkan UNIQUE VALUE NYATA dari kolom segment
# - Menggabungkan variasi penulisan & kombinasi multi-segment
# - Fokus mengurangi "Others" secara signifikan
# =========================================================

import pandas as pd
import numpy as np

# -----------------------------------------------------------------
# 1. Normalisasi awal kolom segment (lowercase, trim, noise removal)
# -----------------------------------------------------------------
df["segment_raw"] = (
    df["segment"]
    .astype(str)
    .str.lower()
    .str.strip()
    .str.replace(r"\s*;\s*", " ; ", regex=True)   # standarisasi delimiter
    .str.replace(r"\s+", " ", regex=True)
)

# Anggap NaN eksplisit
df.loc[df["segment_raw"].isin(["nan", "none", ""]), "segment_raw"] = np.nan

# -----------------------------------------------------------------
# 2. Dictionary segment berbasis UNIQUE VALUE (DATA-DRIVEN)
# -----------------------------------------------------------------
SEGMENT_KEYWORDS = {
    "Pharmaceutical": [
        "farmasi", "pharma", "kedokteran", "biotechnology"
    ],
    "Government": [
        "government"
    ],
    "Agro Industry": [
        "agro", "aggro", "agro industry", "agro/kelapa sawit",
        "kelapa sawit", "feeds"
    ],
    "Education": [
        "education"
    ],
    "Food & Beverages": [
        "food", "beverages"
    ],
    "Laboratory & Testing": [
        "lab service", "laboratory"
    ],
    "Chemical Industry": [
        "chemical", "resin", "plastic", "paint & coating"
    ],
    "Petrochemical & Energy": [
        "petrochemical", "oil", "electricity"
    ],
    "Cosmetic": [
        "cosmetic"
    ],
    "Manufacturing": [
        "manufactur", "manufacture", "packaging", "cement"
    ],
    "Mining": [
        "mining"
    ],
    "Environmental": [
        "enviroment"
    ],
    "Healthcare": [
        "health"
    ],
    "Supplier / Trading": [
        "supplyer"
    ]
}

# ---------------

In [ ]:
# 3. Fungsi klasifikasi segment (multi-keyword aware)
# -----------------------------------------------------------------
def classify_segment_refined(segment):
    if pd.isna(segment):
        return "Unknown"

    matched_segments = []

    for clean_segment, keywords in SEGMENT_KEYWORDS.items():
        for kw in keywords:
            if kw in segment:
                matched_segments.append(clean_segment)
                break

    # Jika multi-segment (mis. "education ; farmasi")
    if len(matched_segments) > 1:
        return " / ".join(sorted(set(matched_segments)))

    # Jika satu segment
    if len(matched_segments) == 1:
        return matched_segments[0]

    # Jika tidak terklasifikasi
    return "Others"

# -----------------------------------------------------------------
# 4. Apply refined segment classification
# -----------------------------------------------------------------
df["segment_clean"] = df["segment_raw"].apply(classify_segment_refined)


In [ ]:
# -----------------------------------------------------------------
# 5. Evaluasi hasil (wajib dijalankan setelah ini)
# -----------------------------------------------------------------
df["segment_clean"].value_counts()

In [ ]:
df_chem_petro = df[
    df["segment_clean"] == "Education"
]

# Jumlah data (validasi ukuran populasi)
df_chem_petro.shape

# Tampilkan sampel acak untuk audit manual
df_chem_petro.sample(
    n=min(10, len(df_chem_petro)),   # Aman jika data < 10
    random_state=42
)

In [ ]:
# =========================================================
# STANDARDISASI PENAMAAN MULTI-SEGMENT
# =========================================================
# Tujuan:
# - Menyamakan urutan penulisan multi-segment
# - Menghindari duplikasi semantik di dashboard
# - Menjaga konsistensi hierarki segment
# =========================================================

# Mapping eksplisit untuk perbaikan nilai tertentu
segment_value_fix = {
    "Chemical Industry / Education": "Education / Chemical Industry"
}

# Apply perbaikan nilai
df["segment_clean"] = df["segment_clean"].replace(segment_value_fix)

# =========================================================
# VALIDASI HASIL
# =========================================================
df["segment_clean"].value_counts()

# =========================================================
# CATATAN BI
# =========================================================
# Langkah ini penting agar:
# - Filter dashboard tidak terpecah
# - Segment analysis tidak terduplikasi
# - Storytelling ke manajemen konsisten
# =========================================================


In [ ]:
# =========================================================
# MERGING MULTI-SEGMENT KE SEGMENT UTAMA
# =========================================================
# Tujuan:
# - Menyederhanakan segmentasi untuk kebutuhan dashboard manajemen
# - Menghindari fragmentasi insight akibat multi-label
# - Memastikan konsistensi KPI per segment
# =========================================================

# Mapping penggabungan nilai segment
segment_merge = {
    "Chemical Industry / Petrochemical & Energy": "Petrochemical & Energy"
}

# Terapkan penggabungan
df["segment_clean"] = df["segment_clean"].replace(segment_merge)

# =========================================================
# VALIDASI HASIL
# =========================================================
df["segment_clean"].value_counts()

# =========================================================
# CATATAN BI
# =========================================================
# Setelah penggabungan:
# - Petrochemical & Energy menjadi segment tunggal yang lebih kuat
# - Chemical Industry fokus pada non-energi
# - Dashboard menjadi lebih mudah dibaca oleh manajemen
# =========================================================


In [ ]:
# 1. Definisikan keyword job_title untuk sub-segment Education
# -----------------------------------------------------------------
EDU_JOB_KEYWORDS = {
    "Education / Pharmaceutical": [
        "farmasi", "pharmacy", "apoteker"
    ],
    "Education / Laboratory": [
        "lab", "laboratorium", "laboran", "analyst", "qc"
    ],
    "Education / Chemical Industry": [
        "chemical", "kimia"
    ],
    "Education / Healthcare": [
        "kedokteran", "medical", "klinik"
    ]
}

# -----------------------------------------------------------------
# 2. Fungsi refinement khusus untuk segment Education
# -----------------------------------------------------------------
def refine_education_segment(row):
    """
    Refinement segment Education berbasis job_title_clean
    """
    if row["segment_clean"] != "Education":
        return row["segment_clean"]

    title = row.get("job_title_clean")

    if pd.isna(title):
        return "Education (General)"

    for refined_segment, keywords in EDU_JOB_KEYWORDS.items():
        if any(kw in title for kw in keywords):
            return refined_segment

    return "Education (General)"

# -----------------------------------------------------------------
# 3. Terapkan refinement hanya pada baris Education
# -----------------------------------------------------------------
df["segment_clean"] = df.apply(refine_education_segment, axis=1)

# -----------------------------------------------------------------
# 4. Validasi hasil refinement
# -----------------------------------------------------------------
df["segment_clean"].value_counts()

# =========================================================
# CATATAN BI
# =========================================================
# Dampak bisnis:
# - Education kini terbagi menjadi sub-segment fungsional
# - Tim sales bisa menentukan pendekatan:
#   • Lab → teknis
#   • Farmasi → regulatori
#   • Healthcare → klinikal
# - Dashboard manajemen menjadi lebih presisi
# =========================================================

##Ekstraksi Wilayah (Geo-Mapping Sederhana)

In [ ]:
import re

In [ ]:
df["address_clean"] = (
    df["address"]
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
city_keywords = {
    # =========================
    # JAWA BARAT (INDUSTRI)
    # =========================
    "Cikarang": ["cikarang", "jababeka", "mm2100", "deltamas"],
    "Karawang": ["karawang", "cikampek", "kiic"],
    "Bekasi": ["bekasi", "cibitung", "tambun"],
    "Bogor": ["bogor", "cibinong", "sentul"],
    "Depok": ["depok", "cilodong"],
    "Bandung": ["bandung", "cimahi"],
    "Sukabumi": ["sukabumi"],
    "Purwakarta": ["purwakarta"],
    "Subang": ["subang"],

    # =========================
    # BANTEN
    # =========================
    "Tangerang": ["tangerang", "cikupa", "balaraja"],
    "Cilegon": ["cilegon"],
    "Serang": ["serang"],

    # =========================
    # JAWA TENGAH & TIMUR
    # =========================
    "Semarang": ["semarang"],
    "Surabaya": ["surabaya"],
    "Gresik": ["gresik", "pasuruan"],
    "Sidoarjo": ["sidoarjo"],
    "Jogja": ["jogja", "yogya", "sleman"],

    # =========================
    # LUAR JAWA
    # =========================
    "Batam": ["batam"],
    "Medan": ["medan"],
    "Palembang": ["palembang"],
    "Balikpapan": ["balikpapan"],
    "Samarinda": ["samarinda"],
    "Makassar": ["makassar"],
    "Denpasar": ["denpasar"],
    "Lombok": ["lombok", "mataram"],
    "Kupang": ["kupang"],
    "Palangkaraya": ["palangkaraya"],

    # =========================
    # JAKARTA (PALING AKHIR)
    # =========================
    "Jakarta": [
        "jakarta pusat",
        "jakarta barat",
        "jakarta timur",
        "jakarta selatan",
        "jakarta utara",
        "pulo gadung",
        "pulogadung",
        "pulo kambing"
        "tanjung priok",
        "cakung", "cikini", "menteng", "pancoran", "pademangan", "kapuk", "fatmawati",
        "jakarta", "cipayung"
    ]
}


In [ ]:
def extract_city(address_clean):
    if pd.isna(address_clean) or address_clean == "":
        return np.nan

    for city, keywords in city_keywords.items():
        for kw in keywords:
            if kw in address_clean:
                return city

    return np.nan



In [ ]:
df["city"] = df["address_clean"].apply(extract_city)


In [ ]:
df["city"].value_counts(dropna=False)


In [ ]:
jakarta_keywords = [
    "jakarta pusat",
    "jakarta barat",
    "jakarta timur",
    "jakarta selatan",
    "jakarta utara",
    "pulo gadung",
    "tanjung priok",
    "cakung"
]

def extract_city_with_jakarta(address_clean):
    city = extract_city(address_clean)
    if pd.notna(city):
        return city

    for kw in jakarta_keywords:
        if kw in address_clean:
            return "Jakarta"

    return np.nan


In [ ]:
mask = df["city"].isna()
df.loc[mask, "city"] = df.loc[mask, "address_clean"].apply(extract_city_with_jakarta)


In [ ]:
df["city"].value_counts(dropna=False)


In [ ]:
df_nan_city = df[df["city"].isna()]

df_nan_city.sample(10)


In [ ]:
df_jakarta = df[df["city"] == "Bogor"]

df_jakarta.sample(10)


In [ ]:
def classify_depok_simple(address):
    if pd.isna(address) or address.strip() == "":
        return "Depok"  # default aman

    addr = address.lower()

    depok_jabar_keywords = [
        "jawa barat", "jabar", "bogor", "bekasi",
        "cilodong", "beji", "pancoran mas",
        "sukmajaya", "cinere", "limo", "bojongsari"
    ]

    depok_jogja_keywords = [
        "yogyakarta", "jogja", "sleman",
        "condongcatur", "maguwoharjo",
        "caturtunggal", "seturan", "karangasem"
    ]

    score_jabar = sum(kw in addr for kw in depok_jabar_keywords)
    score_jogja = sum(kw in addr for kw in depok_jogja_keywords)

    if score_jogja > score_jabar:
        return "Jogja"
    else:
        return "Depok"


In [ ]:
mask_depok = df["city"].str.lower() == "depok"

df.loc[mask_depok, "city"] = df.loc[mask_depok, "address"].apply(classify_depok_simple)


In [ ]:
df["city"].value_counts().loc[["Depok", "Jogja"]]


In [ ]:
df[df["city"].isin(["Depok", "Jogja"])][
    ["company_name", "address", "city"]
].sample(10)


In [ ]:
# =========================================================
# KLASIFIKASI MENDALAM: "JALAN RAYA BOGOR"
# Target output city:
# - Jakarta
# - Depok
# - Bogor
# =========================================================

import re
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Fungsi klasifikasi spesifik Jalan Raya Bogor
# ---------------------------------------------------------
def classify_jalan_raya_bogor(address):
    if pd.isna(address) or address.strip() == "":
        return np.nan

    addr = address.lower()

    # Pastikan hanya case Jalan Raya Bogor
    if not ("jalan raya bogor" in addr or "jl. raya bogor" in addr or "jl raya bogor" in addr):
        return np.nan

    # --- Jakarta Timur (sepanjang Jl Raya Bogor) ---
    jakarta_keywords = [
        "ciracas", "pekayon", "susukan", "rambutan",
        "kramat jati", "pasar rebo", "cijantung",
        "jakarta timur", "jakarta"
    ]

    # --- Depok ---
    depok_keywords = [
        "cimanggis", "cilodong", "tapos",
        "beji", "sukmajaya", "pancoran mas",
        "depok"
    ]

    # --- Bogor ---
    bogor_keywords = [
        "bogor", "ciawi", "tajur",
        "sentul", "gunung putri",
        "cibinong", "sukaraja"
    ]

    score_jakarta = sum(k in addr for k in jakarta_keywords)
    score_depok   = sum(k in addr for k in depok_keywords)
    score_bogor   = sum(k in addr for k in bogor_keywords)

    # Logika keputusan
    if score_jakarta > max(score_depok, score_bogor):
        return "Jakarta"
    elif score_depok > max(score_jakarta, score_bogor):
        return "Depok"
    elif score_bogor > max(score_jakarta, score_depok):
        return "Bogor"
    else:
        # fallback logis:
        # KM besar cenderung Bogor
        km_match = re.search(r"km\s*(\d+)", addr)
        if km_match:
            km = int(km_match.group(1))
            if km <= 15:
                return "Jakarta"
            elif km <= 35:
                return "Depok"
            else:
                return "Bogor"

        # default aman
        return "Bogor"


# ---------------------------------------------------------
# Terapkan ke dataframe
# Hanya overwrite jika city masih NaN / Others / Bogor ambigu
# ---------------------------------------------------------
mask_jrb = (
    df["address"].str.lower().str.contains("jalan raya bogor|jl. raya bogor|jl raya bogor", na=False)
)

df.loc[mask_jrb, "city"] = df.loc[mask_jrb].apply(
    lambda x: classify_jalan_raya_bogor(x["address"])
    if pd.isna(x["city"]) or x["city"].lower() in ["others", "bogor"]
    else x["city"],
    axis=1
)

# ---------------------------------------------------------
# Validasi hasil
# ---------------------------------------------------------
df[mask_jrb]["city"].value_counts()

# Audit sampel
df[mask_jrb][["company_name", "address", "city"]].sample(10)


In [ ]:
df["city"].value_counts(dropna=False)

##Klasifikasi Role Contact Person (Decision Power)

In [ ]:
# =========================================================
# MELIHAT UNIQUE VALUE PADA KOLOM job_title
# =========================================================

# Unique value (sorted, termasuk Unknown)
job_title_unique = (
    df["job_title"]
    .dropna()
    .str.strip()
    .sort_values()
    .unique()
)

# Tampilkan jumlah dan contoh
print(f"Total unique job_title: {len(job_title_unique)}")
job_title_unique


In [ ]:
import pandas as pd
import re

def classify_role(job_title):
    # -------------------------
    # Guard
    # -------------------------
    if pd.isna(job_title):
        return "Unknown"

    jt_raw = str(job_title).strip().lower()

    if jt_raw in ["", "-"]:
        return "Unknown"

    # normalisasi ringan (hanya di variabel lokal)
    jt = re.sub(r"[.;,/]", " ", jt_raw)
    jt = re.sub(r"\s+", " ", jt)

    # =====================================================
    # 1. BUYER / PROCUREMENT (prioritas tertinggi)
    # =====================================================
    if any(k in jt for k in [
        "purchasing", "procurement", "buyer",
        "pengadaan", "logistic", "supply chain"
    ]):
        return "Buyer"

    # =====================================================
    # 2. DECISION MAKER – STRATEGIC
    # =====================================================
    if any(k in jt for k in [
        "direktur", "director", "owner",
        "dekan", "wakil dekan",
        "vice president", "vp",
        "general manager", "gm"
    ]):
        return "Decision Maker"

    # =====================================================
    # 3. DECISION MAKER – LAB / DEPARTMENT HEAD
    # (INI PERBAIKAN UTAMA UNTUK 'Ka. Lab')
    # =====================================================
    if (
        re.search(r"\bka\b", jt) and "lab" in jt
    ) or any(k in jt for k in [
        "kepala lab", "kepala laboratorium",
        "head lab", "head laboratory",
        "manager lab", "manager laboratory",
        "head of qc", "qc head",
        "kaprodi", "kepala upt", "kepala badan"
    ]):
        return "Decision Maker"

    # =====================================================
    # 4. INFLUENCER – OPERATIONAL LEAD
    # =====================================================
    if any(k in jt for k in [
        "spv", "supervisor", "penyelia",
        "koordinator", "kasie",
        "section head", "pj lab",
        "unit head"
    ]):
        return "Influencer"

    # =====================================================
    # 5. USER – TECHNICAL / SCIENTIFIC
    # =====================================================
    if any(k in jt for k in [
        "analyst", "analis", "scientist",
        "laboran", "laboratorium", "lab",
        "peneliti", "researcher",
        "qc", "qa", "validation",
        "engineer", "technician", "teknisi",
        "r&d", "rnd", "mikrobiologi", "kimia"
    ]):
        return "User"

    # =====================================================
    # 6. USER – ACADEMIC / INSTITUTIONAL
    # =====================================================
    if any(k in jt for k in [
        "dosen", "staff kampus", "staff dikampus",
        "balai", "bpom", "poltekkes", "universitas"
    ]):
        return "User"

    # =====================================================
    # 7. FALLBACK
    # =====================================================
    return "Others"


# =========================================================
# APPLY
# =========================================================
df["role_category"] = df["job_title"].apply(classify_role)

# =========================================================
# EVALUASI CEPAT
# =========================================================
df["role_category"].value_counts()

# Audit ulang kasus sensitif
df[df["job_title"].str.contains("lab", case=False, na=False)][
    ["company_name", "job_title", "role_category"]
].sample(15)



In [ ]:
# =========================================================
# VALUE COUNTS job_title
# =========================================================
job_title_counts = df["job_title"].value_counts(dropna=False)

print("=== VALUE COUNTS: job_title ===")
print(job_title_counts)

# (opsional) tampilkan top 30 saja
job_title_counts.head(30)


In [ ]:
# =========================================================
# VALUE COUNTS role_category
# =========================================================
role_category_counts = df["role_category"].value_counts(dropna=False)

print("=== VALUE COUNTS: role_category ===")
print(role_category_counts)


In [ ]:
# Crosstab job_title vs role_category (top job_title saja)
pd.crosstab(
    df["job_title"],
    df["role_category"]
).head(20)


In [ ]:
# Job title yang sering muncul tapi role-nya mencurigakan
df.groupby(["job_title", "role_category"]).size().sort_values(ascending=False).head(30)


In [ ]:
df["role_category"].value_counts(dropna=False)

##Validasi Kontak

In [ ]:
# Flag data yang bisa langsung dihubungi
df["has_phone"] = df["phone"].notna()
df["has_email"] = df["email"].notna()

df["contact_ready"] = np.where(
    (df["has_phone"] == True) | (df["has_email"] == True),
    "Yes",
    "No"
)


In [ ]:
df_dashboard = df[
    [
        "company_name",
        "segment_clean",
        "city",
        "role_category",
        "contact_ready",
        "job_title",
        "phone",
        "email"
    ]
]


In [ ]:
df_dashboard.sample(25)

##Audit Semantik

In [ ]:
def audit_unique(df, col, top_n=30):
    """
    Menampilkan unique value + frekuensi
    Digunakan untuk audit semantik sebelum mapping
    """
    display(
        df[col]
        .fillna("NaN")
        .value_counts()
        .head(top_n)
        .to_frame(name="count")
    )


In [ ]:
audit_unique(df, "segment_clean")


In [ ]:
audit_unique(df, "segment")


In [ ]:
audit_unique(df, "city")


In [ ]:
audit_unique(df, "job_title", top_n=50)


In [ ]:
audit_unique(df, "role_category")


##Missing Value Check

In [ ]:
# =========================================================
# AUDIT MISSING VALUE (DATA QUALITY CHECK)
# =========================================================
# Tujuan:
# - Mengidentifikasi kelengkapan data per kolom
# - Menentukan risiko untuk analisis & dashboard
# - Menjadi dasar rekomendasi data enrichment ke tim sales
# =========================================================

import pandas as pd

# -----------------------------------------------------------------
# 1. Jumlah missing value per kolom
# -----------------------------------------------------------------
missing_count = df.isna().sum().to_frame(name="missing_count")

# -----------------------------------------------------------------
# 2. Persentase missing value per kolom
# -----------------------------------------------------------------
missing_percent = (df.isna().mean() * 100).round(2).to_frame(name="missing_percent")

# -----------------------------------------------------------------
# 3. Gabungkan hasil agar mudah dianalisis
# -----------------------------------------------------------------
missing_summary = missing_count.join(missing_percent)

# -----------------------------------------------------------------
# 4. Urutkan dari kolom paling bermasalah
# -----------------------------------------------------------------
missing_summary = missing_summary.sort_values(
    by="missing_percent",
    ascending=False
)

# Tampilkan ringkasan
missing_summary

# =========================================================
# OPSIONAL: FOKUS PADA KOLOM KRITIS UNTUK MARKETING
# =========================================================
df.isna().sum()

# =========================================================
# INTERPRETASI (PANDUAN BI)
# =========================================================
# - Kolom dengan missing > 30% → risiko tinggi untuk CRM
# - job_title kosong           → sulit klasifikasi decision power
# - phone & email kosong       → prioritas data enrichment
# =========================================================


In [ ]:
audit_unique(df, "city")

In [ ]:
df.sample(10)

#Handling Duplicate

In [ ]:
dup_hard = (
    df[df.duplicated(
        subset=["company_name", "address_clean", "job_title"],
        keep=False
    )]
    .sort_values(["company_name", "address_clean", "job_title"])
)

print("Jumlah hard duplicate:", dup_hard.shape[0])


#Aturan Prioritas Role (WAJIB Eksplisit)

In [ ]:
role_priority = {
    "Decision Maker": 1,
    "Buyer": 2,
    "Influencer": 3,
    "User": 4,
    "Others": 5,
    "Unknown": 6
}


In [ ]:
df["role_priority"] = df["role_category"].map(role_priority)
df["role_priority"] = df["role_priority"].fillna(99)


In [ ]:
# =========================================================
# 1. Urutkan data berdasarkan role_priority (utama dulu)
# =========================================================
df_sorted = df.sort_values("role_priority")

# =========================================================
# 2. Ambil baris UTAMA per entitas
#    (company_name + address_clean)
# =========================================================
df_entity = (
    df_sorted
    .groupby(["company_name", "address_clean"], as_index=False)
    .first()
)

# =========================================================
# 3. Tandai baris utama agar bisa dikecualikan dari alternatif
# =========================================================
df_with_primary = df.merge(
    df_entity[
        ["company_name", "address_clean", "job_title"]
    ].rename(columns={"job_title": "primary_job_title"}),
    on=["company_name", "address_clean"],
    how="left"
)

# =========================================================
# 4. Filter HANYA baris alternatif yang:
#    - company_name & address sama
#    - job_title BERBEDA dari job_title utama
# =========================================================
df_alt = df_with_primary[
    (df_with_primary["job_title"].notna()) &
    (df_with_primary["job_title"] != df_with_primary["primary_job_title"])
]

# =========================================================
# 5. Agregasi kolom ALTERNATIF (tanpa duplikasi identik)
# =========================================================

job_title_alt = (
    df_alt.groupby(["company_name", "address_clean"])["job_title"]
    .apply(lambda x: " | ".join(sorted(set(x))))
    .reset_index(name="job_title_alternative")
)

contact_person_alt = (
    df_alt.groupby(["company_name", "address_clean"])["contact_person"]
    .apply(lambda x: " | ".join(sorted(set(x.dropna()))))
    .reset_index(name="contact_person_alternative")
)

phone_alt = (
    df_alt.groupby(["company_name", "address_clean"])["phone"]
    .apply(lambda x: " | ".join(sorted(set(x.dropna()))))
    .reset_index(name="phone_alternative")
)

email_alt = (
    df_alt.groupby(["company_name", "address_clean"])["email"]
    .apply(lambda x: " | ".join(sorted(set(x.dropna()))))
    .reset_index(name="email_alternative")
)

# =========================================================
# 6. Merge hasil alternatif ke df_entity (entity-level)
# =========================================================
df_entity = (
    df_entity
    .merge(job_title_alt, on=["company_name", "address_clean"], how="left")
    .merge(contact_person_alt, on=["company_name", "address_clean"], how="left")
    .merge(phone_alt, on=["company_name", "address_clean"], how="left")
    .merge(email_alt, on=["company_name", "address_clean"], how="left")
)


In [ ]:
# Tampilkan ringkasan dataframe hasil update
print("Shape df_entity:", df_entity.shape)

# Tampilkan sampel data
df_entity.sample(10)

##Validasi & enrichment kolom kunci

In [ ]:
# ============================================
# FILL NaN PADA KOLOM ALTERNATIVE
# Logika:
# Semua nilai NaN pada kolom *_alternative
# diganti menjadi "Unknown"
# ============================================

cols_alternative = [
    "job_title_alternative",
    "contact_person_alternative",
    "phone_alternative",
    "email_alternative"
]

df_entity = df_entity.copy()

df_entity[cols_alternative] = df_entity[cols_alternative].fillna("Unknown")

In [ ]:
# ============================================
# MARKET PRIORITY (CUSTOM LOGIC – DASHBOARD)
# Fokus: role_priority + contact availability
# ============================================

df_entity = df_entity.copy()

# -----------------------------
# 1️⃣ Base score dari role_priority
# -----------------------------
df_entity["priority_score"] = 0

df_entity.loc[df_entity["role_priority"] == 1, "priority_score"] += 5
df_entity.loc[df_entity["role_priority"] == 2, "priority_score"] += 3
df_entity.loc[df_entity["role_priority"] == 3, "priority_score"] += 2
df_entity.loc[df_entity["role_priority"] >= 4, "priority_score"] += 1


# -----------------------------
# 2️⃣ Contact availability (FIXED)
# -----------------------------
df_entity.loc[df_entity["has_phone"] == True, "priority_score"] += 2
df_entity.loc[df_entity["has_email"] == True, "priority_score"] += 2


# -----------------------------
# 3️⃣ Final market priority label
# -----------------------------
df_entity["market_priority"] = pd.cut(
    df_entity["priority_score"],
    bins=[-1, 4, 8, 20],
    labels=["Low Priority", "Medium Priority", "High Priority"]
)


# =============================
# OUTPUT UNTUK DASHBOARD
# =============================
df_entity[[
    "company_name",
    "address",
    "job_title",
    "contact_person",
    "phone",
    "email",
    "segment_clean", "job_title_alternative",
    "contact_person_alternative",
    "phone_alternative",
    "email_alternative",
    "role_priority",
    "priority_score",
    "market_priority"
]].sort_values(
    by=["priority_score"],
    ascending=False
).sample(20)


##Save File ke CSV

In [ ]:
# ============================================
# SAVE DATAFRAME KE FILE CSV
# ============================================

output_path = "df_entity_final.csv"

df_entity.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"File berhasil disimpan: {output_path}")